
# MIRI LRS slit-less example

This notebook demonstrates the slit-less LRS extension by:

1. building a simple stellar spectrum,
2. inspecting the detector dispersion relation,
3. running a small LRS slit-less cruciform simulation, and
4. plotting the dispersed PSF and cruciform components.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from MIRI_cruciform_diffractio import (
    MIRICruciform,
    lrs_detector_offset_pixels,
    stellar_blackbody_spectrum,
)


In [ ]:

wavelengths = np.linspace(5.0, 12.0, 6)
stellar_weights = stellar_blackbody_spectrum(wavelengths, temperature=6000.0)
detector_offsets = lrs_detector_offset_pixels(wavelengths)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(wavelengths, stellar_weights, marker='o')
axes[0].set_title('Stellar spectrum weights')
axes[0].set_xlabel('Wavelength [$\mu$m]')
axes[0].set_ylabel('Normalised flux')

axes[1].plot(wavelengths, detector_offsets, marker='o')
axes[1].axhline(0.0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('LRS slit-less detector offset')
axes[1].set_xlabel('Wavelength [$\mu$m]')
axes[1].set_ylabel('Offset [pixels]')
plt.tight_layout()


In [ ]:

model = MIRICruciform(mode='LRS-SLTSS', simsize=1024)
components = model.LRSsim(
    wavelengths=wavelengths,
    weights=stellar_weights,
    detector_angle=1.5,
    tr_radius=150.0,
)


In [ ]:

titles = [
    'Total slit-less LRS signal',
    'Direct Webb PSF',
    'Cruciform third pass',
    'Cruciform fifth pass',
]

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for ax, title, component in zip(axes.flat, titles, components):
    positive = component[component > 0]
    vmin = positive.min() if positive.size else 1e-12
    ax.imshow(component, origin='lower', cmap='viridis', norm=LogNorm(vmin=vmin, vmax=component.max()))
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
